# 01 · Data Pipeline
**Mục tiêu**: Gộp dữ liệu lịch sử giá OHLCV + Chỉ số tài chính cơ bản (BCTC quý) + Bảng cân đối kế toán thành bộ dữ liệu hoàn chỉnh trong `data_new/`.

**Chuẩn hóa đơn vị**:
- Giá OHLC trong `data_raw/` lưu ở đơn vị **nghìn đồng** → nhân ×1000 → **đồng**
- Các chỉ số tài chính (`data_transformed/`, `data_balance/`) đã ở đơn vị **đồng** → giữ nguyên

**Chú ý lịch tài chính**:
- **FPT, ELC**: Năm dương lịch tiêu chuẩn (Q1: Jan–Mar, Q2: Apr–Jun, Q3: Jul–Sep, Q4: Oct–Dec)
- **ITD, CMG**: Năm tài chính lệch (Q1: Apr–Jun, Q2: Jul–Sep, Q3: Oct–Dec, Q4: Jan–Mar năm sau)

In [ ]:
# ── 0. Setup ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os, glob

BASE_DIR   = os.path.abspath('')
DATA_RAW   = os.path.join(BASE_DIR, 'data_raw')
DATA_TRANS = os.path.join(BASE_DIR, 'data_transformed')
DATA_BAL   = os.path.join(BASE_DIR, 'data_balance')
DATA_NEW   = os.path.join(BASE_DIR, 'data_new')
os.makedirs(DATA_NEW, exist_ok=True)

TICKERS        = ['FPT', 'CMG', 'ELC', 'ITD']
FISCAL_TICKERS = ['ITD', 'CMG']   # Năm tài chính lệch Apr-Mar
PRICE_COLS     = ['open', 'high', 'low', 'close']  # Cần nhân ×1000

# ── Hàm tiện ích ────────────────────────────────────────────────────────────
def extract_quarter_year(q_str):
    """Parse 'Quý 2/2023' -> (2, 2023)"""
    try:
        parts = str(q_str).replace('Quý', '').strip().split('/')
        if len(parts) != 2:
            return pd.Series([np.nan, np.nan])
        return pd.Series([int(parts[0]), int(parts[1])])
    except:
        return pd.Series([np.nan, np.nan])

def get_quarter_year(df, ticker):
    """Gán cột quarter/year theo lịch tài chính của từng ticker"""
    if ticker in FISCAL_TICKERS:
        # ITD/CMG: Q1=Apr-Jun, Q2=Jul-Sep, Q3=Oct-Dec, Q4=Jan-Mar
        df['quarter'] = df['date'].dt.month.map(
            {4:1, 5:1, 6:1, 7:2, 8:2, 9:2, 10:3, 11:3, 12:3, 1:4, 2:4, 3:4}
        )
        df['year'] = np.where(
            df['date'].dt.month.isin([1, 2, 3]),
            df['date'].dt.year - 1,
            df['date'].dt.year
        )
    else:
        # FPT/ELC: lịch dương lịch chuẩn
        df['quarter'] = df['date'].dt.quarter
        df['year']    = df['date'].dt.year
    return df

print('✅ Setup hoàn tất. Bắt đầu pipeline...')
print(f'   BASE_DIR : {BASE_DIR}')
print(f'   Tickers  : {TICKERS}')

In [ ]:
# ── Bước 1: Merge OHLCV (data_raw) + Fundamental (data_transformed) ─────────
# Chuẩn hóa đơn vị: OHLC nghìn đồng → đồng (×1000)

def merge_fundamental(ticker):
    h_file = os.path.join(DATA_RAW,   f'{ticker}_history.csv')
    t_file = os.path.join(DATA_TRANS, f'{ticker.lower()}_transformed.csv')
    o_file = os.path.join(DATA_NEW,   f'{ticker.lower()}_new.csv')

    # --- Giá OHLCV ---
    df_price = pd.read_csv(h_file)
    df_price.columns = [c.lower() for c in df_price.columns]
    tc = 'time' if 'time' in df_price.columns else 'date'
    df_price.rename(columns={tc: 'date'}, inplace=True)
    df_price['date'] = pd.to_datetime(df_price['date'])
    df_price = df_price.sort_values('date').reset_index(drop=True)

    # ★ Chuyển đơn vị: nghìn đồng → đồng
    for col in PRICE_COLS:
        if col in df_price.columns:
            df_price[col] = df_price[col] * 1000

    df_price = get_quarter_year(df_price, ticker)

    # --- Chỉ số tài chính ---
    df_fund = pd.read_csv(t_file).dropna(axis=1, how='all')
    tc_fund = 'Thời gian' if 'Thời gian' in df_fund.columns else df_fund.columns[0]
    df_fund[['quarter', 'year']] = df_fund[tc_fund].apply(extract_quarter_year)
    df_fund = df_fund.dropna(subset=['quarter', 'year'])
    df_fund[['quarter', 'year']] = df_fund[['quarter', 'year']].astype(int)
    df_fund = df_fund.drop(columns=[tc_fund])

    # --- Merge ---
    merged = pd.merge(df_price, df_fund, on=['year', 'quarter'], how='left')
    merged = merged.sort_values('date').dropna().reset_index(drop=True)
    merged = merged.drop(columns=['year', 'quarter'])
    merged.to_csv(o_file, index=False)

    close_sample = merged['close'].iloc[-1]
    print(f'[+] {ticker}: {len(merged)} rows → {o_file}')
    print(f'    close gần nhất = {close_sample:,.0f} đ  (kiểm tra đơn vị: phải là đồng)')

print('=== Bước 1: Merge OHLCV + Fundamental ===')
for tk in TICKERS:
    merge_fundamental(tk)

In [ ]:
# ── Bước 2: Tích hợp Balance Sheet (data_balance) ────────────────────────────

def integrate_balance(ticker):
    new_file = os.path.join(DATA_NEW, f'{ticker.lower()}_new.csv')
    bal_file = os.path.join(DATA_BAL, f'{ticker.lower()}_balance.csv')

    df_base = pd.read_csv(new_file)
    df_base['date'] = pd.to_datetime(df_base['date'])
    df_base = get_quarter_year(df_base, ticker)

    df_bal = pd.read_csv(bal_file).dropna(subset=['Time'])
    df_bal[['quarter', 'year']] = df_bal['Time'].apply(extract_quarter_year)
    df_bal = df_bal.dropna(subset=['quarter', 'year'])
    df_bal[['quarter', 'year']] = df_bal[['quarter', 'year']].astype(int)
    df_bal = df_bal.drop_duplicates(subset=['year', 'quarter'], keep='last')
    df_bal = df_bal.drop(columns=['Time'])

    merged = pd.merge(df_base, df_bal, on=['year', 'quarter'], how='left')
    merged = merged.sort_values('date').drop(columns=['year', 'quarter']).reset_index(drop=True)
    merged.to_csv(new_file, index=False)
    print(f'[+] {ticker}: balance tích hợp xong, {len(merged)} rows x {len(merged.columns)} cols')

print('=== Bước 2: Tích hợp Balance Sheet ===')
for tk in TICKERS:
    integrate_balance(tk)

In [ ]:
# ── Bước 3: Kiểm tra kết quả cuối ────────────────────────────────────────────
print('=== Kiểm tra data_new/ ===')
print(f'{"Ticker":<8} {"Rows":<8} {"Cols":<6} {"close_min (đ)":<18} {"close_max (đ)":<18}')
print('-' * 65)

for f in sorted(glob.glob(os.path.join(DATA_NEW, '*_new.csv'))):
    df = pd.read_csv(f)
    ticker = os.path.basename(f).replace('_new.csv', '').upper()
    print(f'{ticker:<8} {len(df):<8} {len(df.columns):<6} '
          f'{df["close"].min():<18,.0f} {df["close"].max():<18,.0f}')

print('\n✅ Pipeline hoàn tất! Tất cả file đã được ghi vào data_new/')
print('   Đơn vị giá: đồng (VND) — các chỉ số nợ: đồng (VND) → ĐÃ ĐỒNG NHẤT')